# 03 · Anomaly detection with Gaussian Mixtures (and DBSCAN)

**Method** (Géron, Ch. 9, pp. 256-258, 266-268): fit a Gaussian Mixture Model and treat instances in **low-density** regions as anomalies. The density threshold encodes prior knowledge, e.g. "about 4% of products are defective". The number of components is chosen with the **BIC / AIC**.

The book's examples have no ground-truth anomalies, so here we **inject a known 4% of outliers**. That makes precision and recall measurable and lets us benchmark the GMM against DBSCAN fairly.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from unsupervised_mlops import plots  # shared plot style
from unsupervised_mlops.data import load_digits_split
pd.set_option("display.precision", 4)
split = load_digits_split(random_state=42)
split.sizes

{'n_train': 1347, 'n_test': 450}

In [2]:
from unsupervised_mlops.data import make_blobs_with_outliers, make_moons_with_outliers
from unsupervised_mlops.anomaly import (GaussianMixtureAnomalyDetector, select_gmm, calibrate_dbscan_eps,
                                        DBSCANAnomalyDetector, benchmark_detectors)
Xb, yb = make_blobs_with_outliers(1000, 0.04, min_mahalanobis=3.0, random_state=42)
Xm, ym = make_moons_with_outliers(1000, 0.05, 0.04, min_distance=0.15, random_state=42)
print(f"blobs: {len(Xb)} points, {yb.sum()} injected outliers | moons: {len(Xm)} points, {ym.sum()} injected outliers")

blobs: 1042 points, 42 injected outliers | moons: 1042 points, 42 injected outliers


## Selecting the number of components with BIC / AIC

In [3]:
best, ic = select_gmm(Xb, list(range(1, 11)), n_init=5)
print("BIC picks:", best)
ic.pivot(index="k", columns="covariance_type", values="bic").round(0)

BIC picks: {'n_components': 4, 'covariance_type': 'full'}


covariance_type,diag,full,spherical,tied
k,,,,
1,8594.0,8441.0,8587.0,8441.0
2,8037.0,8001.0,8034.0,8030.0
3,7806.0,7697.0,8043.0,7888.0
4,7840.0,7183.0,7774.0,7907.0
5,7745.0,7220.0,7740.0,7766.0
6,7660.0,7260.0,7633.0,7846.0
7,7594.0,7292.0,7571.0,7587.0
8,7530.0,7327.0,7495.0,7732.0
9,7548.0,7363.0,7479.0,7561.0


## Flag the 4% lowest-density points

In [4]:
from sklearn.metrics import precision_score, recall_score
gmm = GaussianMixtureAnomalyDetector(**best, threshold_percentile=4, n_init=5, random_state=42).fit(Xb)
flag = gmm.predict(Xb) == -1
print(f"flagged {flag.mean():.1%} | precision {precision_score(yb, flag):.3f} | recall {recall_score(yb, flag):.3f}")

flagged 4.0% | precision 0.929 | recall 0.929


## DBSCAN: noise points are anomalies

The book shows how sensitive DBSCAN is to `eps` on the moons (eps = 0.05 finds 7 clusters and many anomalies, eps = 0.2 looks right). To compare it fairly with the GMM, we give it the same prior: we choose the `eps` whose noise fraction is closest to 4%, **without looking at the labels**.

In [5]:
for eps in (0.05, 0.2):
    d = DBSCANAnomalyDetector(eps=eps).fit(Xm)
    print(f"eps={eps}: {len(set(d.labels_)) - (-1 in d.labels_)} clusters, {np.mean(d.labels_ == -1):.1%} flagged")
eps, _ = calibrate_dbscan_eps(Xm, 0.04)
d = DBSCANAnomalyDetector(eps=eps).fit(Xm)
f = d.labels_ == -1
print(f"calibrated eps={eps:.3f}: {f.mean():.1%} flagged | precision {precision_score(ym, f):.3f} | recall {recall_score(ym, f):.3f}")

eps=0.05: 7 clusters, 11.4% flagged
eps=0.2: 2 clusters, 3.5% flagged


calibrated eps=0.150: 4.0% flagged | precision 1.000 | recall 1.000


## Benchmark (one random dataset here; the pipeline averages over 5)

In [6]:
tables = []
for name, (X, y) in {"blobs": (Xb, yb), "moons": (Xm, ym)}.items():
    t, _ = benchmark_detectors(X, y, 0.04, gmm_n_init=5)
    tables.append(t.assign(dataset=name))
pd.concat(tables)[["dataset", "detector", "settings", "precision", "recall", "f1", "roc_auc"]].round(3)

,dataset,detector,settings,precision,recall,f1,roc_auc
0,blobs,GMM (BIC-selected),"k=4, full",0.929,0.929,0.929,0.999
1,blobs,DBSCAN,"eps=0.719, min_samples=5",0.884,0.905,0.894,NaN
2,blobs,Isolation Forest,100 trees,0.905,0.905,0.905,0.988
3,blobs,Local Outlier Factor,n_neighbors=20,0.881,0.881,0.881,0.997
4,blobs,Elliptic Envelope (Fast-MCD),single Gaussian,0.762,0.762,0.762,0.967
0,moons,GMM (BIC-selected),"k=10, full",0.952,0.952,0.952,1.000
1,moons,DBSCAN,"eps=0.150, min_samples=5",1.000,1.000,1.000,NaN
2,moons,Isolation Forest,100 trees,0.738,0.738,0.738,0.971
3,moons,Local Outlier Factor,n_neighbors=20,1.000,1.000,1.000,1.000
4,moons,Elliptic Envelope (Fast-MCD),single Gaussian,0.571,0.571,0.571,0.860


* **GMM** wins on elliptical clusters. On the moons it needs about 9-10 components to approximate the curved shapes (the book's "Oops" moment), but its *density* is still good enough to find outliers.
* **DBSCAN** is near-perfect on the moons, where density-connected shapes are exactly its assumption.
* **Elliptic Envelope (Fast-MCD)** assumes a *single* Gaussian, so it fails on both multi-cluster datasets. This is a useful reminder to check a method's assumptions before using it.

## The detector that ships with the API: digits

64 pixels with full covariances is a lot of parameters for 1,347 images, so we first keep 95% of the variance with PCA (Géron, Ch. 8), then select the GMM by BIC.

In [7]:
from unsupervised_mlops.anomaly import evaluate_on_corruptions
best_d, ic_d = select_gmm(split.X_train, list(range(1, 16)), pca_variance=0.95, n_init=3)
print("BIC picks:", best_d)
common = dict(**best_d, threshold_percentile=4, pca_variance=0.95, n_init=10, random_state=42)
naive = GaussianMixtureAnomalyDetector(**common).fit(split.X_train)
calibrated = GaussianMixtureAnomalyDetector(**common, calibration_fraction=0.25).fit(split.X_train)
for name, det in [("threshold on training densities (book)", naive), ("threshold on held-out densities", calibrated)]:
    print(f"{name:40s}: {np.mean(det.predict(split.X_test) == -1):.1%} of clean test digits flagged (target 4%)")

BIC picks: {'n_components': 6, 'covariance_type': 'full'}


threshold on training densities (book)  : 8.9% of clean test digits flagged (target 4%)
threshold on held-out densities         : 2.9% of clean test digits flagged (target 4%)


Setting the threshold on the **training** densities is optimistic, because the GMM fits its own training data best. On unseen clean digits it raised about twice as many false alarms as intended. Calibrating on a held-out 25% fixes this.

### Does it catch bad inputs?

In [8]:
table, summary = evaluate_on_corruptions(calibrated, split.X_test, random_state=42)
table.round(1)

,input,flagged_pct
0,clean test digits,2.9
1,noise,100.0
2,inverted,100.0
3,shuffled,100.0
4,rotated,100.0
5,mirrored,65.6
6,shifted,78.7
7,gaussian,34.0
8,dimmed,6.0


### Why bother: the classifier cannot say "this is not a digit"

In [9]:
from unsupervised_mlops.clustering_features import build_kmeans_logreg_pipeline
from unsupervised_mlops.data import corrupt_digits
clf = build_kmeans_logreg_pipeline(81, 42, scale_distances=True).fit(split.X_train, split.y_train)
noise = corrupt_digits(split.X_test, "noise", 0)
conf = clf.predict_proba(noise).max(axis=1)
print(f"on pure random noise the classifier's mean confidence is {conf.mean():.0%} "
      f"({np.mean(conf > 0.9):.0%} of noise images get > 90% confidence)")
print(f"the GMM detector flags {np.mean(calibrated.predict(noise) == -1):.0%} of them")

on pure random noise the classifier's mean confidence is 65% (7% of noise images get > 90% confidence)
the GMM detector flags 100% of them


## Takeaways

* A density model gives every prediction a *"do I recognise this input?"* score. The classifier alone cannot: softmax always picks one of the ten digits, and on pure noise it still reports 65% confidence on average.
* The detector catches structural corruptions (noise, inversion, shuffling, rotation) almost perfectly, but **misses most dimmed digits and many mildly noisy ones**. These blind spots are documented in `reports/RESULTS.md` rather than hidden.
* Threshold calibration on held-out data is the difference between a 4% and a 9% false-alarm rate.